### Carga de datos

In [31]:
import librosa
import numpy as np
import warnings

# Suprimimos advertencias de lectura de formatos comprimidos para mantener limpios los logs del ETL
warnings.filterwarnings('ignore', category=UserWarning, module='librosa')

Este módulo utiliza pandas para estructurar la lectura del archivo de texto plano (.txt) proporcionado por el protocolo ASVspoof. Filtramos las columnas redundantes para minimizar la huella de memoria en el entorno de ejecución, conservando exclusivamente la ruta del archivo, la familia del ataque y la etiqueta objetivo.

In [32]:
import pandas as pd
import os

def load_dataset_metadata(protocol_path):
    """
    Carga y filtra el archivo de protocolo del dataset ASVspoof para estructurar las etiquetas.
    
    Argumentos:
    protocol_path: Ruta al archivo .txt de entrenamiento o validación.
    
    Retorna:
    DataFrame de pandas optimizado con las columnas esenciales.
    """
    if not os.path.exists(protocol_path):
        raise FileNotFoundError(f"No se encontró el archivo de protocolo en: {protocol_path}")

    # Definición de la estructura de columnas según el estándar ASVspoof 2019 LA
    columnas = ['speaker_id', 'file_name', 'system_id', 'attack_id', 'label']
    
    # Ingesta del archivo
    df_metadata = pd.read_csv(protocol_path, sep=' ', names=columnas)

    # Filtrado espacial de columnas para optimización de memoria
    df_metadata = df_metadata[['file_name', 'attack_id', 'label']]
    
    # Transformación opcional: Codificación binaria para la red neuronal
    # bonafide (real) -> 0, spoof (IA) -> 1
    # df_metadata['target'] = df_metadata['label'].apply(lambda x: 0 if x == 'bonafide' else 1)

    return df_metadata


ruta_protocolo = '../data/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'
df_etiquetas = load_dataset_metadata(ruta_protocolo)
print(df_etiquetas.head())

      file_name attack_id     label
0  LA_T_1138215         -  bonafide
1  LA_T_1271820         -  bonafide
2  LA_T_1272637         -  bonafide
3  LA_T_1276960         -  bonafide
4  LA_T_1341447         -  bonafide


# ETL: Obtencion de Métricas V2

## Descripcion del enfoque

En esta versión vamos a obtener cambiar el enfoque de la construcción del conjunto de entrenamiento.

En lugar de utilizar la biblioteca librosa para obtener una matriz de métricas de todas las time windows en los que  se ha dividido todo el fichero de audio, y luego almacenar la media y el desvío estandar.

En esta versión vamos a partir de la hipótesis de que es posible clusterizar con suficiente certeza un audio con sólo analizar una porción pequeña. Suponemos que no es necesario analizar una porción tan larga de tiempo, como pueden ser 2000ms. 

En este sentido, comenzaremos por el otro extremo: ananlizaremos las fracciones de tiempo más pequeñas que podemos analizar, siguiendo con el supuesto de que la mejor manera de enfocar el problema es mediante técnicas de sound features luego de la Transformada Rápida de Fourier (FFT).

## Configuración de Parámetros

* n_fft (Tamaño de la Ventana)

Es la cantidad de muestras de audio (samples) que se introducen en el algoritmo de la Transformada Rápida de Fourier (TFFT) para generar un único espectro de frecuencias. Es como el "ancho temporal" de la "foto" que le hace al sonido.

Selección del valor:

El algoritmo funciona de manera óptima solo cuando el tamaño de la entrada es una potencia de 2. Escogemos el número 2048 porque usar un valor que no sea potencia de 2 (como 2000) obligaría al algoritmo a hacer cálculos mucho más lentos.

El Principio de Incertidumbre Acústica: En el procesamiento de audio, si haces la ventana muy corta, se puede saber exactamente cuándo ocurrió un sonido, pero se pierde la capacidad de saber qué frecuencia era (especialmente en los graves). Si la haces muy larga, sabes la frecuencia exacta, pero no sabes en qué milisegundo ocurrió. 2048 muestras a 22,050 Hz (aprox. 93 ms) es un estándar Es lo suficientemente largo para captar las ondas largas de los bajos, pero lo suficientemente corto para no mezclar dos notas musicales rápidas en una sola foto. 

Se podría tambien explorar un valor más alto, para capturar una porción más grande de la muestra.

## Técnicas de Preprosesamiento

Para asegurar la robustez de las métricas extraídas y el correcto entrenamiento del modelo de detección, hemos implementado dos técnicas críticas en la fase inicial de carga de los archivos de audio:

### Remuestreo Explícito

El remuestreo es el proceso mediante el cual convertimos la señal digital de audio desde su frecuencia de muestreo original (fs​) a una frecuencia objetivo estandarizada. En nuestra canalización, forzamos este parámetro estableciendo sr=16000 (16 kHz) al utilizar la función de carga de la librería librosa.

Implementamos esta técnica por los siguientes motivos:

Uniformidad Dimensional: Nuestro conjunto de datos contiene 25,000 archivos con orígenes dispares, lo que implica diferentes tasas de muestreo nativas (por ejemplo, grabaciones bonafide a 8000 Hz frente a audios sintéticos generados a 22050 Hz).

Consistencia del Tiempo Físico: La duración temporal exacta de nuestra ventana de análisis mínima (2048 muestras) depende directamente de la frecuencia de muestreo (T=fs​2048​). Unificar la fs​ a 16000 Hz garantiza que cada ventana extraída represente sistemáticamente el mismo intervalo de tiempo físico (~128 milisegundos) a través de todos los registros.

Viabilidad Algorítmica: Sin un remuestreo explícito, la red neuronal recibiría matrices de Fourier correspondientes a diferentes longitudes temporales relativas. Estandarizar la frecuencia asegura que el modelo compare características equivalentes para identificar correctamente los artefactos de síntesis.

### Normalización de Amplitud (Amplitude Normalization)

La normalización consiste en escalar matemáticamente los valores numéricos de la señal de audio (la forma de onda en formato PCM) a un rango estándar. Por diseño, la carga de datos en punto flotante (float32) que realiza nuestra lógica de extracción escala automáticamente los valores al intervalo continuo de [−1.0,1.0].

Justificamos la aplicación de esta transformación por las siguientes razones:

Invarianza al Volumen: Los audios reales (bonafide) presentan variaciones naturales de volumen dependientes de la ganancia del micrófono y la acústica física. En contraste, las voces generadas sintéticamente suelen entregarse con un nivel de volumen digitalmente constante y normalizado de fábrica.

Prevención de Sesgos (Overfitting): Al normalizar la amplitud general de los audios, eliminamos el volumen global como una variable predictiva. Esto obliga a la red neuronal a no tomar decisiones basadas en fluctuaciones triviales de nivel, forzándola a converger optimizando sus pesos sobre la estructura de los armónicos y las discrepancias de fase acústica, que constituyen las métricas válidas para la detección de spoofing.

## Cantidad de muestras por fichero

Iremos generando diferentes .csv obteniendo muestras con intervalos variables. El objetivo es poder iterar los algoritmos en datasets con espacios variables entre muestras para analizar la distribución óptima.



2. Muestreo Aleatorio Continuo (Random Cropping)

    Explicación: Por cada archivo que pasa por la canalización ETL, se seleccionan N índices de inicio aleatorios (garantizando que no excedan la longitud total menos 2048) y se extraen las ventanas.

    Justificación: Funciona como una técnica inherente de Data Augmentation. Al variar la posición exacta en cada época de entrenamiento (si se hace dinámico) o al extraer una distribución estocástica, obliga al modelo de inteligencia artificial a buscar características fundamentales de spoofing independientemente del contexto fonético, mejorando su capacidad de generalización.

3. Muestreo Basado en Detección de Actividad Vocal (VAD - Energy-based)

    Explicación: Se calcula la energía cuadrática media (RMS) del audio para identificar las regiones de mayor amplitud. Se extraen las ventanas de 2048 muestras exclusivamente en los picos de energía.

    Justificación: Los audios sintéticos suelen presentar mayores dificultades para replicar fielmente las resonancias complejas del tracto vocal humano en las vocales sostenidas (frecuencias formantes). Centrar las "fotografías" en el habla activa maximiza la relación señal-ruido y focaliza el entrenamiento en la estructura armónica del hablante.

4. Muestreo de Transiciones (Onset Detection)

    Explicación: Utilizando funciones de detección de ataques (como librosa.onset.onset_detect), se ubican los cambios abruptos en la envolvente del audio (por ejemplo, el paso de una consonante oclusiva a una vocal). Las ventanas se sitúan exactamente sobre estos eventos.

    Justificación: Los algoritmos de generación de IA a menudo fallan o introducen discontinuidades de fase durante las transiciones acústicas rápidas. Capturar estos momentos críticos proporciona al modelo métricas de Fourier ricas en altas frecuencias temporales, revelando artefactos sintéticos.

5. Muestreo Focalizado en el Suelo de Ruido (Silence/Background)

    Explicación: De forma diametralmente opuesta al método VAD, se extraen las ventanas de los segmentos con la energía no nula más baja, es decir, las pausas intra-silábicas o el inicio/fin del habla.

    Justificación: Las grabaciones reales (Bonafide) poseen una "huella" acústica del entorno (respuesta a impulsos de la sala, ruido térmico del micrófono). Los audios generados por IA tienden a tener un silencio digital antinatural o un ruido de fondo constante y repetitivo carente de las fluctuaciones físicas del mundo real.

6. Muestreo Contiguo Deslizante (Sliding Window Cluster)

    Explicación: En lugar de distribuir las fotos por todo el audio, se selecciona un punto de inicio (aleatorio o de máxima energía) y se extraen N ventanas de 2048 muestras de forma estrictamente secuencial, con un porcentaje de solapamiento (por ejemplo, salto o hop length de 512 muestras).

    Justificación: Al tomar un clúster de ventanas solapadas, se permite calcular derivadas temporales (deltas y delta-deltas de los coeficientes de Fourier). Esto captura la trayectoria espectral a corto plazo, lo cual es vital, ya que la evolución temporal de la fase en señales sintéticas suele ser menos coherente que en las voces reales.

7. Muestreo de Extremos (Head and Tail Extrema)

    Explicación: Se extrae un número fijo de ventanas estrictamente desde el primer milisegundo del archivo y desde el último, ignorando el contenido central.

    Justificación: En la detección de fraude (spoofing), los transitorios de inicio y apagado son reveladores. Un audio bonafide a menudo contiene el "clic" de apertura del canal del micrófono o una respiración inicial. Los modelos generativos suelen iniciar la fonación de manera abrupta o con artefactos de inicio sintético.

8. Muestreo Anclado al Centro (Center-Anchored Symmetrical)

    Explicación: Se localiza el punto temporal exacto a la mitad del archivo de audio (~1000 ms). Se extrae una ventana central, y luego ventanas adicionales dispuestas simétricamente hacia la izquierda y derecha.

    Justificación: Al trabajar con conjuntos de datos donde los silencios de inicio y fin no están estandarizados, el centro del audio es la región estadísticamente más fiable para contener el núcleo de la información articulatoria, eliminando la necesidad de algoritmos de limpieza o recorte previos.

9. Muestreo por Máxima Variación Espectral (Spectral Flux Maximization)

    Explicación: Durante el proceso ETL, se calcula rápidamente el Flujo Espectral (la tasa de cambio entre espectros adyacentes a lo largo del archivo). Se seleccionan aquellas ventanas de 2048 muestras donde el flujo espectral sea máximo.

    Justificación: Esto dirige la atención de la red hacia los momentos de mayor complejidad acústica y entropía. Reproducir fielmente altas tasas de cambio espectral es computacionalmente costoso para los modelos de síntesis de voz, por lo que estas áreas son candidatas ideales para buscar discrepancias generativas.

10. Muestreo de Ráfagas Periódicas (Burst / Stratified Clustering)

    Explicación: Una combinación de métodos. Se divide el audio de 2000 ms en, por ejemplo, tres macrorregiones. En cada macrorregión, se extrae una "ráfaga" de tres ventanas muy juntas (solapadas).

    Justificación: Proporciona lo mejor de ambos enfoques: una cobertura global del archivo de audio (para entender el contexto macro) y micro-contexto temporal en puntos específicos (mediante las ráfagas) para que el modelo identifique inconsistencias de fase a muy corto plazo.

## Módulo de Carga y Preprocesamiento

El primer paso lógico de nuestra ETL es la ingesta de datos. En esta función aplicamos el remuestreo explícito y la normalización de amplitud. Esto nos garantiza que, independientemente del origen del audio (y mitigando los efectos del bitrate de 189 Kbps), todos los tensores ingresen al sistema con la misma frecuencia de muestreo (fs​=16000 Hz) y en el rango continuo de [−1.0,1.0].

In [33]:
def load_and_preprocess_audio(file_path, target_sr=16000):
    """
    Ingesta el archivo de audio aplicando remuestreo explícito y normalización.
    """
    audio_time_series, sr = librosa.load(file_path, sr=target_sr)
    
    return audio_time_series, target_sr

# ETL 1: Muestreo Uniforme Estricto (Equidistante)

Explicación: Consiste en dividir la duración total del audio en N segmentos iguales y extraer una ventana de 2048 muestras en el centro de cada segmento.

Justificación: Es el enfoque base más robusto. Garantiza que la red neuronal reciba información de todas las fases temporales del audio (inicio, desarrollo y final) sin introducir sesgos algorítmicos. Evita que la red se sobreajuste a una sola parte de la fonación.

Punto negativo: No nos es posible saber con exactitud la distancia entre muestras, lo que puede dificultar su posterior testeo en muestras desconocidas. 

Hemos incluido una lógica estricta de control de bordes para evitar errores de desbordamiento de memoria (IndexError) en los extremos del archivo.

## Métricas a Obtener

### Dimensionalidad del Tensor (Características Matemáticas)

Para cada "fotografía" o segmento de 2048 muestras temporales, la Transformada Rápida de Fourier a Corto Plazo (STFT) seguida del cálculo del valor absoluto (np.abs(stft_matrix))  dedevuelve un vector columna tridimensional.

Dado un tamaño de ventana N=2048, la transformada de Fourier de una señal real produce un espectro simétrico. Por eficiencia algorítmica, librosa.stft descarta la mitad redundante y retorna un número de contenedores de frecuencia o bins igual a 2N​+1.

Entonces Por cada ventana extraída, obtenemos exactamente 1025 características (features). Si configuramos la ETL para extraer 5 ventanas por archivo, la matriz final por audio tendrá una dimensión de (5,1025).

### Significado Físico: El Espectro de Magnitud

Las métricas obtenidas representan el espectro de magnitud lineal.

Originalmente, la STFT devuelve números complejos (a+bi), donde la magnitud representa la amplitud de la onda y el ángulo representa la fase. Al aplicar el valor absoluto, descartamos intencionalmente la información de fase topológica.

Cada uno de los 1025 valores resultantes indica la cantidad de energía acústica presente en una banda de frecuencia específica (desde 0 Hz hasta la frecuencia de Nyquist, que es 2fs​​, es decir, 8000 Hz) durante esa fracción de 128 milisegundos.

### Relevancia Predictiva para la Detección de Spoofing

Justificamos la selección de este vector de 1025 variables como input para nuestro modelo de inteligencia artificial basándonos en la naturaleza de los ataques de síntesis de voz (Text-to-Speech y Voice Conversion):

Detección de Artefactos de Vocoder: Los modelos generativos que construyen el audio final (vocoders) a menudo introducen anomalías estructurales en las altas frecuencias (ruido de cuantización o armónicos artificiales). Estas anomalías son invisibles en el dominio del tiempo, pero aparecen como picos anómalos de energía en bins específicos de nuestro vector de 1025 características.

Mapeo de Formantes: La distribución de energía en las frecuencias bajas y medias mapea las resonancias físicas del tracto vocal humano (formantes). Los algoritmos de IA a menudo producen espectros excesivamente "suavizados" en estas áreas, careciendo de la micro-rugosidad acústica de la fonación natural. Nuestra red neuronal detectará estos patrones de suavizado en las magnitudes de los bins.

Reducción de Dimensionalidad Implícita: Al transformar 2048 muestras temporales altamente ruidosas en 1025 contenedores de frecuencia ordenados, proporcionamos a la red neuronal un espacio latente mucho más estructurado, facilitando la convergencia de los pesos durante el entrenamiento por descenso de gradiente.

Consideramos que se trata de una cantidad interesante como vector de entrada.

## Formato de Salida

El resultado nativo de nuestra canalización ETL no es una tabla bidimensional estándar, sino un tensor tridimensional.

Al procesar el conjunto de datos completo, la matriz resultante (nuestra variable independiente X) posee tres ejes o dimensiones fundamentales:

* Dimensión 0 (Muestras - Samples): Representa el número total de archivos de audio que han pasado por la ETL (por ejemplo, 25,000 audios).

* Dimensión 1 (Pasos Temporales - Time Steps/Frames): Representa la cantidad de "fotografías" o ventanas analíticas extraídas por cada archivo de audio (en nuestra configuración actual, N=5 ventanas).

* Dimensión 2 (Características - Features/Bins): Representa la resolución espectral obtenida tras aplicar la STFT sobre cada ventana de 2048 muestras (1025 contenedores de frecuencia).

En conjunto, la dimensión matricial global de nuestro conjunto de datos es (25000, 5, 1025).

## Definición de Funcinoes

### Selección de muestras

In [34]:
def calculate_uniform_segments(total_samples, n_frames=5, frame_length=2048):
    """
    Calcula los índices de inicio y fin para un muestreo equidistante sobre el eje temporal.
    """
    if total_samples < frame_length:
        raise ValueError(f"Longitud de audio ({total_samples}) inferior a la ventana NFFT ({frame_length}).")

    segment_size = total_samples // n_frames
    indices = []

    for i in range(n_frames):
        # Determinamos el centro matemático del segmento
        segment_center = (i * segment_size) + (segment_size // 2)
        
        # Proyectamos la ventana de 2048 muestras
        start = segment_center - (frame_length // 2)
        end = start + frame_length
        
        # Control de bordes para garantizar tensores de tamaño constante
        if start < 0:
            start = 0
            end = frame_length
        if end > total_samples:
            end = total_samples
            start = end - frame_length
            
        indices.append((int(start), int(end)))
        
    return indices

### Módulo de Transformación al Dominio de la Frecuencia

Una vez aislada la ventana temporal (el frame), aplicamos la Transformada Rápida de Fourier a Corto Plazo (STFT). Al extraer el valor absoluto (np.abs), descartamos la información de fase pura y conservamos la magnitud del espectro, que es la métrica principal que nuestra red neuronal utilizará para identificar anomalías y artefactos de los vocoders.

In [35]:
def extract_fourier_metrics(audio_frame, frame_length=2048):
    """
    Transforma un segmento temporal de audio a su representación de magnitud espectral.
    """
    # Aplicamos STFT. El hop_length mayor al frame evita cálculos redundantes.
    stft_matrix = librosa.stft(audio_frame, n_fft=frame_length, hop_length=frame_length+1)
    magnitude_spectrum = np.abs(stft_matrix)
    
    return magnitude_spectrum

### Orquestador del Pipeline (Función Principal)

Finalmente, consolidamos la arquitectura. Esta función actúa como el controlador de la ETL, invocando secuencialmente los módulos lógicos definidos anteriormente. Es el único punto de entrada que llamaremos al iterar sobre nuestros 25,000 archivos.

In [36]:
def etl_uniform_extraction_pipeline(file_path, n_frames=5, frame_length=2048, target_sr=16000):
    """
    Orquestador principal de la ETL para extracción de métricas de Fourier.
    """
    # 1. Preprocesamiento
    audio, sr = load_and_preprocess_audio(file_path, target_sr)
    
    # 2. Cálculo espacial
    total_samples = len(audio)
    frame_indices = calculate_uniform_segments(total_samples, n_frames, frame_length)
    
    # 3. Extracción y Transformación
    features_matrix = []
    
    for start, end in frame_indices:
        # Aislamiento del vector temporal
        audio_frame = audio[start:end]
        
        # Transformación a Fourier
        fourier_metrics = extract_fourier_metrics(audio_frame, frame_length)
        features_matrix.append(fourier_metrics)
        
    # Retornamos un tensor tridimensional listo para la ingesta del modelo de IA
    return np.array(features_matrix), frame_indices

### Ejecución del Procesamiento por Lotes (Batch Processing) y Agregación de Tensores

Con la arquitectura lógica definida y los metadatos estructurados, procedemos a la fase de ejecución intensiva. En esta etapa, iteramos sobre nuestro conjunto de datos documentado para procesar cada archivo de audio a través del pipeline ETL.

Para gestionar la carga computacional y la monitorización del progreso sobre los archivos, hemos incorporado la librería tqdm. Además, hemos implementado un bloque de control de excepciones para garantizar que la canalización no se detenga abruptamente si encuentra un archivo corrupto o faltante. Finalmente, los tensores resultantes (X para las características de Fourier e y para las etiquetas) se serializan y almacenan en disco para su uso posterior en la fase de entrenamiento de la red neuronal.

Ejecución secuencial del pipeline sobre el corpus de audio. Transformamos las etiquetas de texto en formato binario continuo (0 para bonafide, 1 para spoof) para mantener la compatibilidad con las funciones de pérdida estándar de las redes neuronales.

In [37]:
# --- Celda 11 Refactorizada: Paralelización con Joblib ---
import os
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed
import multiprocessing

AUDIO_DIR_PATH = '../data/LA/ASVspoof2019_LA_train/flac/'
CORES_DISPONIBLES = multiprocessing.cpu_count()

def process_single_file_joblib(row, audio_dir, n_frames):
    """
    Worker serializable por loky/joblib para el procesamiento atómico.
    """
    file_name = row['file_name']
    label = row['label']
    file_path = os.path.join(audio_dir, f"{file_name}.flac")
    label_binary = 0 if label == 'bonafide' else 1
    
    try:
        # Llamada al orquestador principal
        features_matrix, _ = etl_uniform_extraction_pipeline(
            file_path, 
            n_frames=n_frames, 
            frame_length=2048, 
            target_sr=16000
        )
        return {"status": "success", "features": features_matrix, "label": label_binary, "file": file_name}
    except Exception as e:
        return {"status": "error", "error": str(e), "file": file_name}

def run_joblib_etl(df_metadata, audio_dir, n_frames=5):
    """
    Orquesta la ejecución paralela resolviendo el problema de BrokenProcessPool en Windows.
    """
    print(f"Iniciando extracción paralela (Joblib) de {len(df_metadata)} audios...")
    print(f"Distribuyendo carga en {CORES_DISPONIBLES} núcleos lógicos.")

    # Ejecución paralela con barra de progreso integrada
    resultados = Parallel(n_jobs=CORES_DISPONIBLES)(
        delayed(process_single_file_joblib)(row, audio_dir, n_frames) 
        for _, row in tqdm(df_metadata.iterrows(), total=df_metadata.shape[0])
    )

    # Consolidación de los diccionarios resultantes
    X_features = [res["features"] for res in resultados if res["status"] == "success"]
    y_labels = [res["label"] for res in resultados if res["status"] == "success"]
    archivos_fallidos = [res for res in resultados if res["status"] == "error"]

    X_tensor = np.array(X_features)
    y_tensor = np.array(y_labels)

    print("\nPipeline ETL finalizado.")
    print(f"Dimensiones de Matriz de Características (X): {X_tensor.shape}")
    print(f"Dimensiones de Vector de Etiquetas (y): {y_tensor.shape}")
    print(f"Errores de ingesta: {len(archivos_fallidos)}")

    return X_tensor, y_tensor, archivos_fallidos

### Modulo de guardado en Disco

In [ ]:
def save_tensors_to_disk(X_tensor, y_tensor, output_dir='./processed_data'):
    """
    Serializa y almacena los tensores resultantes en almacenamiento persistente.
    """
    # Creacion del directorio si no existe
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    x_path = os.path.join(output_dir, 'X_fourier_features.npy')
    y_path = os.path.join(output_dir, 'y_labels.npy')
    
    np.save(x_path, X_tensor)
    np.save(y_path, y_tensor)
    
    print(f"Datos guardados exitosamente en:\n - {x_path}\n - {y_path}")

## Bloque de Ejecución

In [40]:
# --- Comando de Ejecución Final ---
if __name__ == '__main__':
    # Ejecutamos la arquitectura refactorizada
    X_final, y_final, errores = run_joblib_etl(df_etiquetas, AUDIO_DIR_PATH, n_frames=5)
    
    # Invocamos la Celda 9 para persistir en formato .npy
    save_tensors_to_disk(X_final, y_final, output_dir='../Metricas/ETL_V2.1')

Iniciando extracción paralela (Joblib) de 25380 audios...
Distribuyendo carga en 20 núcleos lógicos.


100%|██████████| 25380/25380 [00:04<00:00, 5634.47it/s]



Pipeline ETL finalizado.
Dimensiones de Matriz de Características (X): (25380, 5, 1025, 1)
Dimensiones de Vector de Etiquetas (y): (25380,)
Errores de ingesta: 0
Datos guardados exitosamente en:
 - ../Metricas/ETL_V2.1\X_fourier_features.npy
 - ../Metricas/ETL_V2.1\y_labels.npy
